> **Tarihî defter (protokol 1.3, zero-shot).** İki zero-shot final'i 22 Kasım 2026'da bitti. Protokol 1.4 sonrası bu defter çalıştırılmaz; few-shot deneyi için `notebooks/fewshot.ipynb` kullanılır.


# Fine-tune: kontrol ve SAR varyantı

İnce defter: bütün mantık `src/hasar` paketinde. Ana (SAR artırımı olmayan kontrol) ve varyant (SAR radyometrik
artırımlı) aynı DamageFormer zero-shot checkpoint'inden başlar. Doldurulacak bir şey yok; defter git etiketlerini
okuyup aşamasını kendi seçer ve iki koşuyu sırayla çalıştırır:

- `protokol-kilidi` etiketi yoksa yalnız ön kontrol çalışır (eğitim yok).
- `protokol-kilidi` varsa önce ön kontrol (zaten geçtiyse atlanır), ardından fine-tune çalışır (`last.pt` varsa
  kaldığı yerden devam eder, `DONE` varsa atlanır).
- `varyant-kilidi` de varsa her koşunun eğitiminden sonra `ornekler`, `sicaklik` ve tek Türkiye değerlendirmesi
  (`final`) çalışır.

Bağlantı koparsa yeniden "Run All" yeterlidir.
Ayrıntı: `docs/tasarim.md` Bölüm 8, `docs/colab.md`.


In [ ]:
# Hücre 1: Drive, aşama seçimi (git etiketleri), kurulum, sırlar
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

STEPS_OVERRIDE = None  # ör. 2000 (kısa prova); asıl eğitimde None
RUNS = [("configs/ana.yaml", "ana"), ("configs/varyant.yaml", "varyant")]

drive.mount("/content/drive")
# Bellek parçalanmasını azaltır; alt süreçler (hasar komutları) de bu ayarı devralır.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPO = Path("/content/deprem-hasar-haritasi")
DRIVE = Path("/content/drive/MyDrive/deprem-hasar")
DATA = Path("/content/data/bright")
CKPT = Path("/content/data/checkpoints")
OUT = DRIVE / "kosular"


def sh(cmd, check=True):
    """Run a shell command and stream its output into the cell (Colab hides plain subprocess output)."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc:
        raise subprocess.CalledProcessError(rc, cmd)
    return rc


if not REPO.exists():
    sh(f"git clone https://github.com/OgzhnTarhn/deprem-hasar-haritasi.git {REPO}")
sh(f"cd {REPO} && git fetch --tags --force --quiet origin")
# Önceki oturumun klonu kirli kalmış olabilir; etiketli commit'e geçmeden önce sıfırla.
sh(f"cd {REPO} && git reset --quiet --hard")
tags = subprocess.run(f"cd {REPO} && git tag -l", shell=True, capture_output=True, text=True).stdout
TAGS = {line.strip() for line in tags.splitlines() if line.strip()}
LOCKED = "protokol-kilidi" in TAGS
RUN_FINAL = "varyant-kilidi" in TAGS
# Her zaman en güncel main kullanılır (defter ve kod düzeltmeleri etiket taşımadan gelir); kilitli dosyalar
# etiketten beri değişmemiş olmalı. `final` ayrıca iki etiketin HEAD'in atası olduğunu kendisi denetler.
REF = "origin/main"
sh(f"cd {REPO} && git checkout --quiet --force {REF}")
dirty = subprocess.run(f"cd {REPO} && git status --porcelain", shell=True, capture_output=True, text=True).stdout
assert not dirty.strip(), "Çalışma ağacı kirli; commit değiştirilmemeli."


def locked_files_changed(tag, paths):
    """Return the files under ``paths`` that differ between ``tag`` and HEAD (or a note if not an ancestor)."""
    anc = subprocess.run(f"cd {REPO} && git merge-base --is-ancestor {tag} HEAD", shell=True).returncode
    if anc != 0:
        return [f"{tag} HEAD'in atası değil"]
    out = subprocess.run(
        f"cd {REPO} && git diff --name-only {tag} HEAD -- {paths}", shell=True, capture_output=True, text=True
    ).stdout
    return [line for line in out.splitlines() if line.strip()]


if LOCKED:
    changed = locked_files_changed("protokol-kilidi", "configs/ana.yaml configs/varyant.yaml docs/protokol.md")
    assert not changed, f"protokol-kilidi'nden sonra kilitli dosya değişmiş: {changed}"
if RUN_FINAL:
    changed = locked_files_changed("varyant-kilidi", "configs/")
    assert not changed, f"varyant-kilidi'nden sonra yapılandırma değişmiş: {changed}"
COMMIT = subprocess.run(f"cd {REPO} && git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
if RUN_FINAL:
    STAGE = "ön kontrol + eğitim + Türkiye final"
elif LOCKED:
    STAGE = "ön kontrol + eğitim"
else:
    STAGE = "ön kontrol yalnız"
print("=" * 72)
print("AŞAMA:", STAGE)
print("REF:", REF)
print("commit:", COMMIT)
print("=" * 72)
# Colab'ın torch ve timm sürümleri kullanılır; torch yeniden kurulmaz.
sh(f"pip install -q -e {REPO} einops")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN sırrı yok; indirme anonim yapılır.")
DRIVE.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hücre 2: Ortam kapısı (A100 ya da L4).
import shutil

import torch

props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 2**30, 1), "GB")
# Colab A100 seçilse de bazen L4 (22 GB) veriyor; ikisi de bf16 destekler. Bellek yetmezse eğitim mikro partiye iner.
assert any(gpu in props.name for gpu in ("A100", "L4")), f"A100 ya da L4 gerekli, gelen: {props.name}"
free_gb = shutil.disk_usage("/content").free / 2**30
assert free_gb >= 60, f"Disk yetersiz: {free_gb:.0f} GB"
print("torch", torch.__version__, "CUDA", torch.version.cuda)

In [ ]:
# Hücre 3: Veri (VM diskine) ve iki koşunun başlangıç ağırlığı
sh(f"cd {REPO} && hasar indir --root {DATA} --cache /content/data/cache")  # .done varsa atlanır


def sh_out(cmd):
    """Run a shell command, print its output and return stdout on success."""
    print("$", cmd, flush=True)
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(proc.stdout, end="", flush=True)
    print(proc.stderr, end="", flush=True)
    proc.check_returncode()
    return proc.stdout


PRETRAINED = {}
for config, name in RUNS:
    output = sh_out(f"cd {REPO} && hasar agirlik --config {config} --out {CKPT}")  # md5 denetimli
    lines = [line.strip() for line in output.splitlines() if line.strip()]
    assert lines, f"Başlangıç ağırlığının yolu yazılmadı: {name}"
    PRETRAINED[name] = Path(lines[-1])
    assert PRETRAINED[name].is_file(), f"Başlangıç ağırlığı bulunamadı: {name}: {PRETRAINED[name]}"

In [ ]:
# Hücre 4: Ortak yeniden üretim (kapı 1) ve her koşunun ön kontrol, eğitim, final akışı
import json

REPRO = REPO / "results" / "yeniden-uretim"
for key in ["siamcrnn", "unet", "damageformer"]:
    if not (REPRO / f"{key}_test.json").exists():
        sh(f"cd {REPO} && hasar yeniden-uret --model {key} --data {DATA} --out {REPRO} --cache {CKPT} --device cuda")


def run_one(config, name):
    pre_dir = OUT / "onkontrol" / name
    pre = pre_dir / "preflight.json"
    steps = f"--steps {STEPS_OVERRIDE}" if STEPS_OVERRIDE else ""
    # Ön kontrol zaten geçtiyse kendini atlar; bu yüzden her "Run All"da çağrılır.
    rc = sh(
        f"cd {REPO} && hasar on-kontrol --config {config} --data {DATA} --out {pre_dir} "
        f"--device cuda --pretrained {PRETRAINED[name]} --repro-dir {REPRO}",
        check=False,
    )
    failed = []
    if pre.exists():
        report = json.loads(pre.read_text(encoding="utf-8"))
        for gate in report.get("gates", []):
            verdict = "geçti" if gate.get("passed") else "KALDI"
            print(f"kapı {gate.get('gate')}  {verdict}  {gate.get('detail', '')}")
            if not gate.get("passed"):
                failed.append(gate.get("gate"))
            if gate.get("gate") == 9:
                print("kırpma eşiği (clip):", gate.get("value", {}).get("clip"))
    if rc != 0:
        raise AssertionError(
            f"Ön kontrol kapılarından biri geçmedi ({name}, kapılar: {failed}); eğitim başlamaz.\n"
            f"preflight.json: {pre}\nBu dosyayı Claude'a gönderin."
        )
    if not LOCKED:
        print(f"Ön kontrol geçti ({name}). Eğitim `protokol-kilidi` etiketini bekliyor (Claude atacak).")
        return
    sh(
        f"cd {REPO} && hasar egit --config {config} --data {DATA} --out {OUT} "
        f"--device cuda --pretrained {PRETRAINED[name]} --commit {COMMIT} --preflight {pre} {steps}"
    )
    if RUN_FINAL:
        res = DRIVE / "sonuclar" / name
        INST = Path("/content/data/ornekler")
        CACHE = Path("/content/data/final_cache")
        sh(f"cd {REPO} && hasar ornekler --dest {INST} --cache /content/data/cache")  # yalnız sabit 1.001 kare
        if not (res / "sicaklik.json").exists():  # sıcaklık yalnız 13 olayın doğrulamasında seçilir
            sh(f"cd {REPO} && hasar sicaklik --config {config} --data {DATA} --runs {OUT} --device cuda --out {res}")
        if (res / "turkiye.json").exists():
            print("turkiye.json var: Türkiye değerlendirmesi zaten yapıldı, ikinci kez çalışmaz.")
        else:
            sh(
                f"cd {REPO} && hasar final --config {config} --data {DATA} --runs {OUT} --device cuda "
                f"--out {res} --cache {CACHE} --instances {INST}"
            )
        if CACHE.exists():  # yeni oturumda final atlandıysa yerel önbellek yoktur
            sh(f"mkdir -p {DRIVE / 'final_cache'} && cp -rn {CACHE}/. {DRIVE / 'final_cache'}/")
        print("Sonuçlar:", res)
    else:
        print("Türkiye değerlendirmesi atlandı: `varyant-kilidi` etiketi yok ya da eğitim bu oturumda çalışmadı.")

In [ ]:
# Hücre 5: Kontrol fine-tune (last.pt varsa devam eder, DONE varsa atlanır)
run_one(*RUNS[0])

In [ ]:
# Hücre 6: SAR varyantı (aynı başlangıç, ayrı ön kontrol ve sonuçlar)
run_one(*RUNS[1])

In [ ]:
# Hücre 7: Kopyaları doğrula ve çalışma zamanını bırak (CU yakımı durur)
from google.colab import runtime

done = list(OUT.glob("*/DONE"))
print("DONE:", [str(p) for p in done])
for p in done:
    print(json.loads(p.read_text(encoding="utf-8")))
runtime.unassign()